# The income column is an ID in disguise

Playground S6E9 asks you to predict a binary
outcome over 668,665 training rows, scored by roc_auc. Like every Playground dataset it is synthetic: a generative
model was trained on a real dataset and sampled to produce these rows. That
sentence is usually treated as trivia. This notebook treats it as the most
important fact in the data.

Here is why. `Annual_Income_USD` looks like a continuous feature. It is not behaving like
one. Every one of its values is a whole number, and only 13,214
distinct values occur across 668,665 rows. When a "continuous" column
reuses exact values dozens of times each, those values stop being measurements
and start being labels. The generator has a limited repertoire, and each exact
value points back at a small cluster of the original data.

That makes an exact value a partial row identifier. A tree cannot exploit it -
range splits treat two adjacent values as neighbours. Target encoding treats them
as different keys, and on this dataset encoding this one column moved
cross-validated roc_auc by about **+0.003**, roughly twenty times the
accept bar this notebook measures for itself.

It also moved the public leaderboard by +0.0037. That is the one number here no
cell can reproduce - there is no internet in a competition kernel - so treat it
as a submission record and nothing more. Every *other* number below is printed by
a cell you can run.

## Setup

LightGBM and a deliberately modest configuration, so this runs quickly enough to fork and rerun.

In [ ]:
import glob, os
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

# Find the data rather than hardcoding a path. Kaggle has moved competition data
# between /kaggle/input/<slug>/ and /kaggle/input/competitions/<slug>/, and a
# notebook that hardcodes either one breaks without warning when it changes.
found = sorted(glob.glob("/kaggle/input/**/*train*.csv", recursive=True))
if not found:
    raise FileNotFoundError(
        "no training csv under /kaggle/input - attach the competition data via "
        "'+ Add Input' in the sidebar")
DATA, TRAIN = os.path.dirname(found[0]), found[0]
TARGET, ID = "Will_Buy_EV", "id"
print("data:", TRAIN)

train = pd.read_csv(TRAIN)

# The target is usually numeric, but not always. Everything below takes means of
# it, so encode once here. Sorted rather than first-seen, so the mapping is a
# property of the data and not of the row order.
CLASSES = None
if not pd.api.types.is_numeric_dtype(train[TARGET]):
    CLASSES = sorted(train[TARGET].dropna().unique().tolist(), key=str)
    train[TARGET] = train[TARGET].map({c: i for i, c in enumerate(CLASSES)})
    print(f"target was categorical: {CLASSES} -> 0..{len(CLASSES) - 1}")

print(f"{train.shape[0]:,} rows x {train.shape[1]} columns")
train.head(3)

In [ ]:
FEATURES = [c for c in train.columns if c not in (TARGET, ID)]
CAT = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(train[c])]

def prep(df):
    X = df[FEATURES].copy()
    for c in CAT:
        X[c] = X[c].fillna("__MISSING__").astype("category")
    return X

def score(y, p):
    return roc_auc_score(y, p)

METRIC = "roc_auc"
print(f"{len(FEATURES)} features, {len(CAT)} categorical, metric {METRIC}")

In [ ]:
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,
              min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8,
              bagging_freq=1, verbose=-1, n_jobs=-1)
N_TREES = 400

def cv_score(X, y, seed, n_splits=5):
    """One full cross-validation under seed `seed` - partition AND model.

    The seed moves the PARTITION and the model's own randomness together. Pinning
    the model seed and varying only the partition gives a floor about ten times
    too small - it measures one source of run-to-run variation and calls it all of
    them. A bar built that way then marks trivial deltas as real wins.
    """
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for i, (tr, va) in enumerate(kf.split(X, y)):
        m = lgb.train(dict(PARAMS, seed=seed * 100 + i),
                      lgb.Dataset(X.iloc[tr], y[tr]), num_boost_round=N_TREES)
        oof[va] = m.predict(X.iloc[va])
    return score(y, oof), oof

y = train[TARGET].values
X = prep(train)

## The bar

Fold-to-fold spread is not your uncertainty. It tells you how much the score
moves across *subsets of your data*. It says almost nothing about how much the
fold **mean** would move if you had split the data differently - and that second
quantity is the one an idea has to beat.

So measure it directly: run the entire cross-validation again under a different
seed - moving the partition **and** the model's own randomness together, because
both vary between two runs of your pipeline - several times, and take the
standard deviation of the means.

Three seeds gives a rough estimate: the std of a std from three samples carries
roughly 50% relative error, so treat the bar as an order of magnitude, not a
threshold to three significant figures.

In [ ]:
means = []
for s in [0, 1, 2]:
    m, _ = cv_score(X, y, seed=s)
    means.append(m)
    print(f"partition seed {s}: {m:.6f}")

floor = float(np.std(means))
BAR = 2 * floor
print(f"\nnoise floor (std of the means) = {floor:.6f}")
print(f"accept bar (2x)                = {BAR:.6f}")

## 1. Granularity census

For every feature: how many distinct values, what fraction are whole numbers, and how many rows sit behind each level.

In [ ]:
rows = []
for c in FEATURES:
    s = train[c]
    nun = s.astype("string").nunique()
    isint = float((s.dropna() % 1 == 0).mean()) if pd.api.types.is_numeric_dtype(s) else np.nan
    rows.append({"column": c, "levels": nun, "pct_integer": isint,
                 "rows_per_level": len(train) / max(nun, 1)})
cen = pd.DataFrame(rows).sort_values("levels", ascending=False)
print(cen.to_string(index=False, formatters={
    "levels": lambda v: f"{v:,}", "pct_integer": lambda v: f"{v:.0%}" if v == v else "-",
    "rows_per_level": lambda v: f"{v:,.1f}"}))

## 2. The mode structure of `Annual_Income_USD`

In [ ]:
C = "Annual_Income_USD"
s = train[C]
vc = s.value_counts()
print(f"range {s.min():,.0f} - {s.max():,.0f}   {s.nunique():,} distinct values")
span = int(s.max() - s.min()) + 1
print(f"possible integers in that range: {span:,}   occupancy {100*s.nunique()/span:.1f}%")
print(f"\ntop 10 values:")
for v, n in vc.head(10).items():
    print(f"  {v:>12,.0f}  {n:>7,} rows  {100*n/len(train):5.2f}%")
print(f"\nvalues appearing exactly once: {(vc == 1).sum():,}")

The most common value is very likely a floor or clip in the original data rather
than a discriminative mode - read it as a boundary, not as signal.

## 3. Do exact levels carry signal beyond chance?

If a level has `n` rows and the overall rate is `p`, then with no signal at all
its mean is a binomial average with variance `p(1-p)/n`. Compare that to the
variance actually observed across levels. **The level filter is printed with the
numbers** - the ratio changes a lot depending on which levels you admit.

In [ ]:
MIN_ROWS = 20
p = float(y.mean())
g = train.groupby(train[C].astype("string"), observed=True)[TARGET].agg(["mean", "size"])
g = g[g["size"] >= MIN_ROWS]
obs = float(g["mean"].var())
exp = float((p * (1 - p) / g["size"]).mean())
print(f"levels with >= {MIN_ROWS} rows: {len(g):,}")
print(f"  target rate  min {g['mean'].min():.3f}  median {g['mean'].median():.3f}  max {g['mean'].max():.3f}")
print(f"  observed std of level means      {obs**0.5:.4f}")
print(f"  binomial-noise std if NO signal  {exp**0.5:.4f}")
print(f"  ratio  {(obs/exp)**0.5:.1f}x in std, {obs/exp:.1f}x in variance")

### The exhibit: near-identical values, opposite outcomes

In [ ]:
gg = g.copy()
gg.index = gg.index.astype(float)
gg = gg.sort_index()
big = gg[gg["size"] >= 100]
vals = big.index.to_numpy()
pairs = []
for i in range(len(vals) - 1):
    for j in range(i + 1, min(i + 6, len(vals))):
        if vals[j] - vals[i] > 500:
            break
        d = abs(big['mean'].iloc[i] - big['mean'].iloc[j])
        if d > 0.5:
            pairs.append((vals[i], big['size'].iloc[i], big['mean'].iloc[i],
                          vals[j], big['size'].iloc[j], big['mean'].iloc[j], d))
pairs.sort(key=lambda t: -t[-1])
print(f"pairs within 500 units, both >= 100 rows, target rates differing > 0.5: {len(pairs)}")
for a, na, ra, b, nb, rb, d in pairs[:5]:
    print(f"  {a:>10,.0f} ({na:>4,} rows) rate {ra:.3f}   vs "
          f"{b:>10,.0f} ({nb:>4,} rows) rate {rb:.3f}   gap {d:.3f}")

No real income effect changes an outcome probability by fifty points across a few
hundred dollars. A generator reproducing modes from its training data does it
routinely. That is the whole diagnosis.

## 4. Does it reach the test set?

In [ ]:
test = pd.read_csv(sorted(glob.glob(f"{DATA}/*test*.csv"))[0])
ktr = train[C].astype("string")
kte = test[C].astype("string")
vc2 = ktr.value_counts()
seen = kte.isin(vc2.index)
sup = kte.map(vc2).fillna(0)
print(f"test rows whose {C} also appears in train: {100*seen.mean():.2f}%")
print(f"  median train rows behind a test row's value: {sup.median():.0f}")
print(f"  (higher than the average rows/level because common values are")
print(f"   over-represented among test ROWS - size-biased sampling)")
fcols = [c for c in FEATURES if c in test.columns]
print(f"\nexact duplicate feature rows in train: {100*train.duplicated(subset=fcols).mean():.2f}%")

So it is not row duplication, and the encoding will be just as well populated at
inference as it is in cross-validation.

## 5. The trap: the obvious ranking statistic gets it wrong

Excess variance looks like the statistic for choosing which column to encode. It
is not sufficient, and on this dataset it ranks the candidates backwards.

In [ ]:
p = float(y.mean())
rows = []
for c in FEATURES:
    gg = train.groupby(train[c].astype("string"), observed=True)[TARGET].agg(["mean", "size"])
    gg = gg[gg["size"] >= 20]
    if len(gg) < 2:
        continue
    o = float(gg["mean"].var()); e = float((p * (1 - p) / gg["size"]).mean())
    rows.append({"column": c, "levels": train[c].astype("string").nunique(),
                 "signal_ratio_std": (o / e) ** 0.5})
sr = pd.DataFrame(rows).sort_values("signal_ratio_std", ascending=False)
print("levels >= 20 rows only:")
print(sr.to_string(index=False, formatters={
    "levels": lambda v: f"{v:,}", "signal_ratio_std": lambda v: f"{v:.1f}"}))

The top of that table is strong ordinary features - of course a two-level flag
with two very different rates has enormous level-mean variance. And the two
high-cardinality candidates come out as near-twins. One of them is worth +0.003
and the other is worth nothing. The statistic is not misranking them; it is
**blind to the thing that matters**. It measures predictive strength, and never
asks whether the model already has the signal.

Here is a statistic that can see it. LightGBM histograms every feature into 255
bins, and two values inside one bin are indistinguishable to the model - so any
target signal *within* a bin is invisible to a range split, and exact-value
pooling is the only thing that collects it. So bin each column the way the model
does, and measure how much level means vary around **their own bin's mean**,
against binomial noise.

In [ ]:
# More levels than the model's histogram has bins: that is the whole condition.
# 255 is lightgbm's max_bin default, so this is the mechanism, not a round number.
MAX_BIN = 255
HI = sr[sr["levels"] > MAX_BIN]["column"].tolist()
print(f"columns with more levels than max_bin={MAX_BIN}: {HI}")
for c in HI:
    bins = pd.qcut(train[c], q=MAX_BIN, duplicates="drop")
    d = pd.DataFrame({"b": bins, "v": train[c].astype("string"), "y": y})
    lev = d.groupby(["b", "v"], observed=True)["y"].agg(["mean", "size"])
    lev = lev[lev["size"] >= 20].join(
        d.groupby("b", observed=True)["y"].agg(bm="mean", bn="size"), on="b")
    obs = float(((lev["mean"] - lev["bm"]) ** 2).mean())
    exp = float((p * (1 - p) * (1 / lev["size"] - 1 / lev["bn"])).mean())
    print(f"  {c:26s} within-bin ratio {(obs / exp) ** 0.5:.2f}"
          f"   (1.0 = nothing left inside a bin)")

One column comes out near 1.0: inside a bin its levels are interchangeable, so
its excess variance is a smooth function of the underlying quantity and the
baseline already has all of it. The other comes out around 2 - several times
binomial variance still surviving inside bins the model cannot subdivide.

That is identity signal, and the next section is the receipt.

## 6. The receipt

Three arms, identical folds, one variable: which column gets encoded. Encoding is
nested out-of-fold, so no row contributes to its own encoding.

In [ ]:
def enc_map(keys, yv, prior, smoothing=20.0):
    gg = pd.DataFrame({"k": keys, "y": yv})
    agg = gg.groupby("k")["y"].agg(["sum", "count"])
    return (agg["sum"] + prior * smoothing) / (agg["count"] + smoothing)

def cv_te(cols, seed=0, n_splits=5, inner=5):
    oof = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in skf.split(X, y):
        Xtr, Xva = X.iloc[tr].copy(), X.iloc[va].copy()
        ytr = y[tr]; prior = float(ytr.mean())
        for c in cols:
            keys = train[c].astype("string").fillna("__MISSING__").to_numpy()
            ktr, kva = keys[tr], keys[va]
            full = enc_map(ktr, ytr, prior)
            Xva[f"te_{c}"] = pd.Series(kva).map(full).astype(float).fillna(prior).values
            colv = np.full(len(ktr), prior, dtype=float)
            ikf = StratifiedKFold(n_splits=inner, shuffle=True, random_state=seed)
            for itr, iva in ikf.split(ktr, ytr):
                m = enc_map(ktr[itr], ytr[itr], prior)
                colv[iva] = pd.Series(ktr[iva]).map(m).astype(float).fillna(prior).values
            Xtr[f"te_{c}"] = colv
        mdl = lgb.train(dict(PARAMS, seed=0), lgb.Dataset(Xtr, ytr),
                        num_boost_round=N_TREES)
        oof[va] = mdl.predict(Xva)
    return score(y, oof)

In [ ]:
# The contrast must be a column with MANY levels and a high ratio - not the
# top-ranked column overall, which is a 2-level flag a range split reaches for
# free. The whole point is that ratio without cardinality misleads.
mid = sr[(sr.levels > 255) & (sr.column != C)]   # 255 = lightgbm max_bin
RUNNER_UP = mid.iloc[0]["column"] if len(mid) else sr.iloc[1]["column"]
print(f"contrast column: {RUNNER_UP} "
      f"({int(sr.loc[sr.column == RUNNER_UP, 'levels'].iloc[0]):,} levels)")
arms = [("A  baseline, no encoding", []),
        (f"B  encode {C}", [C]),
        (f"C  encode {RUNNER_UP}", [RUNNER_UP])]
res = {}
for name, cols in arms:
    res[name] = cv_te(cols)
    print(f"{name:34s} {METRIC}={res[name]:.6f}")

base = res["A  baseline, no encoding"]
idea_score = res[f"B  encode {C}"]
print()
print(f"{'arm':34s}{'score':>11}{'delta':>12}   vs bar {BAR:.6f}")
for name, s in res.items():
    d = s - base
    v = "-" if name.startswith("A") else ("WIN" if d > BAR else
        "HARMFUL" if d < -BAR else "inside the noise")
    print(f"{name:34s}{s:>11.6f}{d:>+12.6f}   {v}")

## What to take from it

On a new Playground dataset, run this before you model. For each numeric column
print three numbers: distinct-value count, percent integer-valued, and the excess
variance of per-level target rates over what binomial noise alone would produce,
using only levels with at least twenty rows.

Then apply the rule this notebook earned: **the raw variance ratio measures
strength, not encodability.** Here it scored the +0.003 column and the worthless
one within ten percent of each other.

For every column with more levels than your model's histogram bins, bin it the
way the model does and measure the excess variance of level means *within* bins.
That number is the signal only exact-value pooling can collect, and it separated
the two candidates that the raw ratio called twins.

Encode where three things coincide: within-bin ratio well above one, more levels
than the histogram resolves, and high overlap between train and test values.

And keep the finding in its lane. This is a fact about the generator, not about
the people in the data. On a real dataset a column that behaves this way is a
leak to delete before it embarrasses you. On synthetic data it is the
leaderboard, and everyone who has not encoded it is leaving the gain on the
table.